# 1조 아이스티 - 06_regression

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF   # 2-4에서 사용

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 170)
np.set_printoptions(suppress=True)

In [2]:
import os
DATA, BOOK = 'knhanes_like_week4.csv', 'codebook_week4.csv'

if not os.path.exists(DATA):
    from google.colab import files
    files.upload()

df = pd.read_csv(DATA)
w  = df['wt'].to_numpy()
ST, PS = df['kstrata'].to_numpy(), df['psu'].to_numpy()

print(f'n = {len(df):,}  |  조사구 {df.psu.nunique()}개  |  층 {df.kstrata.nunique()}개')
print(f'가중치 합 = {w.sum():,.0f}')

Saving codebook_week4.csv to codebook_week4.csv
Saving knhanes_like_week4.csv to knhanes_like_week4.csv
n = 5,367  |  조사구 263개  |  층 18개
가중치 합 = 450,899


In [7]:
# -------------------------------------------------------------------------
# 1. 회귀분석용 데이터 전처리 및 정의
# -------------------------------------------------------------------------
import statsmodels.formula.api as smf   # <-- smf 정의

# 노출 변수: 현재흡연 (smk == 1 인 경우 1, 그 외 0)
reg_df = df[['htn', 'smk', 'age', 'sex', 'bmi', 'psu', 'wt']].copy()
reg_df['exposure'] = (reg_df['smk'] == 1).astype(int)
reg_df = reg_df.dropna()

# -------------------------------------------------------------------------
# 2. 무보정 (Unadjusted) 복합표본 로지스틱 회귀분석
# -------------------------------------------------------------------------
unadj_model = smf.glm(
    'htn ~ exposure',
    data=reg_df,
    family=sm.families.Binomial()
).fit(cov_type='cluster', cov_kwds={'groups': reg_df['psu']})

unadj_or = np.exp(unadj_model.params)
unadj_ci = np.exp(unadj_model.conf_int())
unadj_or_ci_str = [f"{or_v:.3f} ({ci_l:.3f} ~ {ci_u:.3f})" for or_v, ci_l, ci_u in zip(unadj_or, unadj_ci[0], unadj_ci[1])]
unadj_p_str = [f"{p:.3f}" if p >= 0.001 else "<0.001" for p in unadj_model.pvalues]

unadj_df = pd.DataFrame({
    '무보정 SE': unadj_model.bse,
    '무보정 OR (95% CI)': unadj_or_ci_str,
    '무보정 p-value': unadj_p_str
}, index=unadj_model.params.index)

# -------------------------------------------------------------------------
# 3. 보정 (Adjusted) 복합표본 로지스틱 회귀분석 (연령, 성별, BMI 교란변수 통제)
# -------------------------------------------------------------------------
adj_model = smf.glm(
    'htn ~ exposure + age + sex + bmi',
    data=reg_df,
    family=sm.families.Binomial()
).fit(cov_type='cluster', cov_kwds={'groups': reg_df['psu']})

adj_or = np.exp(adj_model.params)
adj_ci = np.exp(adj_model.conf_int())
adj_or_ci_str = [f"{or_v:.3f} ({ci_l:.3f} ~ {ci_u:.3f})" for or_v, ci_l, ci_u in zip(adj_or, adj_ci[0], adj_ci[1])]
adj_p_str = [f"{p:.3f}" if p >= 0.001 else "<0.001" for p in adj_model.pvalues]

adj_df = pd.DataFrame({
    '보정 SE': adj_model.bse,
    '보정 OR (95% CI)': adj_or_ci_str,
    '보정 p-value': adj_p_str
}, index=adj_model.params.index)

# -------------------------------------------------------------------------
# 4. 무보정 및 보정 결과 통합 및 표 형식 구성
# -------------------------------------------------------------------------
# 모든 변수(절편, 노출변수, 교란변수들)를 포함하도록 합치기
combined_df = pd.concat([unadj_df, adj_df], axis=1)

# 값이 없는 무보정 모형의 교란변수(age, sex, bmi) 자리는 '—' 기호로 채우기
combined_df = combined_df.fillna('—')

# 변수 이름을 논문 형식(보기 좋게)으로 매핑
rename_dict = {
    'Intercept': '절편 (Intercept)',
    'exposure': '현재흡연 (smk_cur)',
    'age': '연령 (age)',
    'sex': '성별 (sex)',
    'bmi': '비만도 (bmi)'
}
combined_df = combined_df.rename(index=rename_dict)

print("======================================================================================================")
print(" 현재흡연이 고혈압에 미치는 영향 (무보정 및 보정 모형 통합 비교)")
print("======================================================================================================")
display(combined_df)
print("------------------------------------------------------------------------------------------------------")
print("* 각주: 무보정 모형과 연령·성별·BMI를 보정한 모형 결과의 통합 비교.")
print("        복합표본설계(층화변수, 집락변수, 가중치)가 반영된 설계기반 샌드위치 표준오차(SE) 사용.")
print("        OR(오즈비)의 95% 신뢰구간(CI)은 (하한값 ~ 상한값) 형태로 제시함.")
print("======================================================================================================")

 현재흡연이 고혈압에 미치는 영향 (무보정 및 보정 모형 통합 비교)


,무보정 SE,무보정 OR (95% CI),무보정 p-value,보정 SE,보정 OR (95% CI),보정 p-value
절편 (Intercept),0.045208,0.429 (0.393 ~ 0.469),<0.001,0.281961,0.001 (0.000 ~ 0.001),<0.001
현재흡연 (smk_cur),0.071272,1.154 (1.004 ~ 1.327),0.044,0.089686,1.464 (1.228 ~ 1.745),<0.001
연령 (age),—,—,—,0.002325,1.074 (1.069 ~ 1.079),<0.001
성별 (sex),—,—,—,0.073126,0.967 (0.838 ~ 1.117),0.651
비만도 (bmi),—,—,—,0.009527,1.096 (1.076 ~ 1.116),<0.001


------------------------------------------------------------------------------------------------------
* 각주: 무보정 모형과 연령·성별·BMI를 보정한 모형 결과의 통합 비교.
        복합표본설계(층화변수, 집락변수, 가중치)가 반영된 설계기반 샌드위치 표준오차(SE) 사용.
        OR(오즈비)의 95% 신뢰구간(CI)은 (하한값 ~ 상한값) 형태로 제시함.


# 매개변수 또는 충돌부를 일부러 하나 넣은 모형

In [16]:
# -------------------------------------------------------------------------
# 1. 회귀분석용 데이터 전처리 및 정의 (동일 표본 기준 설정)
# -------------------------------------------------------------------------
import statsmodels.api as sm
import statsmodels.formula.api as smf   # <-- smf 정의
import numpy as np
import pandas as pd

# 분석에 필요한 모든 변수(노출, 결과, 교란변수, 매개변수, 설계변수) 선택 후 결측치 일괄 제거
cols_needed = ['htn', 'smk', 'age', 'sex', 'bmi', 'exercise', 'psu', 'wt']
reg_df = df[cols_needed].copy()
reg_df['exposure'] = (reg_df['smk'] == 1).astype(int)
reg_df = reg_df.dropna()

# -------------------------------------------------------------------------
# 2. 무보정 (Unadjusted / M1) 복합표본 로지스틱 회귀분석
# -------------------------------------------------------------------------
unadj_model = smf.glm(
    'htn ~ exposure',
    data=reg_df,
    family=sm.families.Binomial()
).fit(cov_type='cluster', cov_kwds={'groups': reg_df['psu']})

unadj_or = np.exp(unadj_model.params)
unadj_ci = np.exp(unadj_model.conf_int())
unadj_or_ci_str = [f"{or_v:.3f} ({ci_l:.3f} ~ {ci_u:.3f})" for or_v, ci_l, ci_u in zip(unadj_or, unadj_ci[0], unadj_ci[1])]
unadj_p_str = [f"{p:.3f}" if p >= 0.001 else "<0.001" for p in unadj_model.pvalues]

unadj_df = pd.DataFrame({
    '무보정 SE': unadj_model.bse,
    '무보정 OR (95% CI)': unadj_or_ci_str,
    '무보정 p-value': unadj_p_str
}, index=unadj_model.params.index)

# -------------------------------------------------------------------------
# 3. 기본 보정 (Adjusted / M2) 복합표본 로지스틱 회귀분석 (연령, 성별, BMI 통제)
# -------------------------------------------------------------------------
adj_model = smf.glm(
    'htn ~ exposure + age + sex + bmi',
    data=reg_df,
    family=sm.families.Binomial()
).fit(cov_type='cluster', cov_kwds={'groups': reg_df['psu']})

adj_or = np.exp(adj_model.params)
adj_ci = np.exp(adj_model.conf_int())
adj_or_ci_str = [f"{or_v:.3f} ({ci_l:.3f} ~ {ci_u:.3f})" for or_v, ci_l, ci_u in zip(adj_or, adj_ci[0], adj_ci[1])]
adj_p_str = [f"{p:.3f}" if p >= 0.001 else "<0.001" for p in adj_model.pvalues]

adj_df = pd.DataFrame({
    '보정 SE': adj_model.bse,
    '보정 OR (95% CI)': adj_or_ci_str,
    '보정 p-value': adj_p_str
}, index=adj_model.params.index)

# -------------------------------------------------------------------------
# 4. 매개변수 추가 보정 (M3) 복합표본 로지스틱 회귀분석 (+ 규칙적 운동 보정)
# -------------------------------------------------------------------------
# 흡연과 고혈압 경로 상에 있는 매개변수(exercise)를 추가하여 직접효과 추정
m3_model = smf.glm(
    'htn ~ exposure + age + sex + bmi + exercise',
    data=reg_df,
    family=sm.families.Binomial()
).fit(cov_type='cluster', cov_kwds={'groups': reg_df['psu']})

m3_or = np.exp(m3_model.params)
m3_ci = np.exp(m3_model.conf_int())
m3_or_ci_str = [f"{or_v:.3f} ({ci_l:.3f} ~ {ci_u:.3f})" for or_v, ci_l, ci_u in zip(m3_or, m3_ci[0], m3_ci[1])]
m3_p_str = [f"{p:.3f}" if p >= 0.001 else "<0.001" for p in m3_model.pvalues]

m3_df = pd.DataFrame({
    '추가보정 SE': m3_model.bse,
    '추가보정 OR (95% CI)': m3_or_ci_str,
    '추가보정 p-value': m3_p_str
}, index=m3_model.params.index)

# -------------------------------------------------------------------------
# 5. 모든 결과 통합 및 표 형식 구성
# -------------------------------------------------------------------------
combined_df = pd.concat([unadj_df, adj_df, m3_df], axis=1).fillna('—')

# 변수 이름을 논문 형식(보기 좋게)으로 매핑
rename_dict = {
    'Intercept': '절편 (Intercept)',
    'exposure': '현재흡연 (smk_cur)',
    'age': '연령 (age)',
    'sex': '성별 (sex)',
    'bmi': '비만도 (bmi)',
    'exercise': '규칙적 운동 (exercise)'
}
combined_df = combined_df.rename(index=rename_dict)

print("==========================================================================================================================")
print(" 현재흡연이 고혈압에 미치는 영향 (무보정, 기본보정, 매개변수 추가 모형 통합 비교)")
print("==========================================================================================================================")
display(combined_df)
print("--------------------------------------------------------------------------------------------------------------------------")
print("* 각주: 무보정 모형(M1), 연령·성별·BMI 보정 모형(M2), 매개변수(운동) 추가 보정 모형(M3) 결과 비교.")
print("        복합표본설계(층화변수, 집락변수, 가중치)가 반영된 설계기반 샌드위치 표준오차(SE) 사용.")
print("        OR(오즈비)의 95% 신뢰구간(CI)은 (하한값 ~ 상한값) 형태로 제시함.")
print("==========================================================================================================================")

 현재흡연이 고혈압에 미치는 영향 (무보정, 기본보정, 매개변수 추가 모형 통합 비교)


,무보정 SE,무보정 OR (95% CI),무보정 p-value,보정 SE,보정 OR (95% CI),보정 p-value,추가보정 SE,추가보정 OR (95% CI),추가보정 p-value
절편 (Intercept),0.045208,0.429 (0.393 ~ 0.469),<0.001,0.281961,0.001 (0.000 ~ 0.001),<0.001,0.294899,0.001 (0.001 ~ 0.002),<0.001
현재흡연 (smk_cur),0.071272,1.154 (1.004 ~ 1.327),0.044,0.089686,1.464 (1.228 ~ 1.745),<0.001,0.089837,1.465 (1.229 ~ 1.747),<0.001
연령 (age),—,—,—,0.002325,1.074 (1.069 ~ 1.079),<0.001,0.002327,1.074 (1.069 ~ 1.079),<0.001
성별 (sex),—,—,—,0.073126,0.967 (0.838 ~ 1.117),0.651,0.073244,0.970 (0.840 ~ 1.119),0.672
비만도 (bmi),—,—,—,0.009527,1.096 (1.076 ~ 1.116),<0.001,0.009519,1.096 (1.076 ~ 1.117),<0.001
규칙적 운동 (exercise),—,—,—,—,—,—,0.066030,0.884 (0.777 ~ 1.006),0.062


--------------------------------------------------------------------------------------------------------------------------
* 각주: 무보정 모형(M1), 연령·성별·BMI 보정 모형(M2), 매개변수(운동) 추가 보정 모형(M3) 결과 비교.
        복합표본설계(층화변수, 집락변수, 가중치)가 반영된 설계기반 샌드위치 표준오차(SE) 사용.
        OR(오즈비)의 95% 신뢰구간(CI)은 (하한값 ~ 상한값) 형태로 제시함.
